#  Fraudulent Transactions Data Analysis
##  Exploratory Data Analysis (EDA) & Data Preparation

###  Methodology: CRISP-DM Framework
This project is structured based on the **Cross-Industry Standard Process for Data Mining (CRISP-DM)** framework to ensure a comprehensive and business-aligned approach to fraud detection. 

This specific notebook covers the first three phases of the methodology:

- [x] **1. Business Understanding:** Defining the project objectives and requirements from a business perspective to tackle financial fraud.
- [x] **2. Data Understanding:** Initial data collection, exploratory data analysis (EDA), and identification of patterns and data quality issues.
- [x] **3. Data Preparation:** Data cleaning, handling missing values, and feature engineering to prepare the dataset for predictive modeling.
- [ ] **4. Modeling:** (To be developed in a separate notebook/script).
- [ ] **5. Evaluation:** Assessing model performance metrics against business goals.
- [ ] **6. Deployment:** (Future scope).

## 1. Business Understanding
Financial fraud is a growing challenge for financial institutions, leading to direct monetary losses, increased operational costs, and damage to customer trust. In mobile money environments, malicious agents often attempt to compromise accounts, transfer funds, and cash out before being detected.

Project Goal: The primary objective of this project is to develop a robust, data-driven risk analysis pipeline. By identifying hidden patterns in transactional data, we aim to build a predictive model that accurately flags fraudulent activities (CASH-OUT and TRANSFER operations) in real-time.

Business Value: A successful model will minimize financial exposure and optimize the fraud detection process, focusing on a high detection rate while strictly controlling false positives to ensure legitimate users are not negatively impacted.

## 1. Importing the libraries

In [1]:
import polars as pl
import plotly.express as px
from pathlib import Path
import sys

import fraud_detection.visualizer as vis

# 2. Understanding the data

* The dataset was collected from Kaggle: https://www.kaggle.com/datasets/chitwanmanchanda/fraudulent-transactions-data/data

* This dataset is derived from the renowned **PaySim** synthetic dataset. PaySim is a financial mobile money simulator designed to generate synthetic data that closely resembles the normal operation of transactions, while mathematically injecting malicious behavior to evaluate fraud detection methods.
The underlying simulation is based on a sample of real transactions extracted from one month of financial logs from a mobile money service implemented in an African country.

 **Original Academic Citation:**
*E. A. Lopez-Rojas, A. Elmir, and S. Axelsson. "PaySim: A financial mobile money simulator for fraud detection". In: The 28th European Modeling and Simulation Symposium-EMSS, Larnaca, Cyprus. 2016.*

In [2]:
data_path = '/data/PROJETOS/ALURA-FRAUDE/data/Fraud.csv'
df = pl.read_csv(data_path)

In [3]:
df.head()

step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
i64,str,f64,str,f64,f64,str,f64,f64,i64,i64
1,"""PAYMENT""",9839.64,"""C1231006815""",170136.0,160296.36,"""M1979787155""",0.0,0.0,0,0
1,"""PAYMENT""",1864.28,"""C1666544295""",21249.0,19384.72,"""M2044282225""",0.0,0.0,0,0
1,"""TRANSFER""",181.0,"""C1305486145""",181.0,0.0,"""C553264065""",0.0,0.0,1,0
1,"""CASH_OUT""",181.0,"""C840083671""",181.0,0.0,"""C38997010""",21182.0,0.0,1,0
1,"""PAYMENT""",11668.14,"""C2048537720""",41554.0,29885.86,"""M1230701703""",0.0,0.0,0,0


## **Data Dictionary**

1. **step** - maps a unit of time in the real world. In this case 1 step is 1 hour of time. Total steps 744 (30 days simulation).

2. **type** - CASH-IN, CASH-OUT, DEBIT, PAYMENT and TRANSFER.

3. **amount** - amount of the transaction in local currency.

4. **nameOrig** - customer who started the transaction

5. **oldbalanceOrg** - initial balance before the transaction

6. **newbalanceOrig** - new balance after the transaction

7. **nameDest** - customer who is the recipient of the transaction

8. **oldbalanceDest** - initial balance recipient before the transaction. Note that there is not information for customers that start with M (Merchants).

9. **newbalanceDest** - new balance recipient after the transaction. Note that there is not information for customers that start with M (Merchants).

10. **isFraud** - This is the transactions made by the fraudulent agents inside the simulation. In this specific dataset the fraudulent behavior of the agents aims to profit by taking control or customers accounts and try to empty the funds by transferring to another account and then cashing out of the system.

11. **isFlaggedFraud** - The business model aims to control massive transfers from one account to another and flags illegal attempts. An illegal attempt in this dataset is an attempt to transfer more than 200.000 in a single transaction.

In [4]:
df.describe()

statistic,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
str,f64,str,f64,str,f64,f64,str,f64,f64,f64,f64
"""count""",6.36262e6,"""6362620""",6.36262e6,"""6362620""",6.36262e6,6.36262e6,"""6362620""",6.36262e6,6.36262e6,6.36262e6,6.36262e6
"""null_count""",0.0,"""0""",0.0,"""0""",0.0,0.0,"""0""",0.0,0.0,0.0,0.0
"""mean""",243.397246,null,179861.903549,null,833883.104074,855113.668579,null,1.1007e6,1.2250e6,0.001291,0.000003
"""std""",142.331971,null,603858.231463,null,2.8882e6,2.9240e6,null,3.3992e6,3.6741e6,0.035905,0.001586
"""min""",1.0,"""CASH_IN""",0.0,"""C1000000639""",0.0,0.0,"""C1000004082""",0.0,0.0,0.0,0.0
"""25%""",156.0,null,13389.57,null,0.0,0.0,null,0.0,0.0,0.0,0.0
"""50%""",239.0,null,74872.08,null,14208.0,0.0,null,132705.81,214661.65,0.0,0.0
"""75%""",335.0,null,208721.45,null,107315.0,144258.41,null,943036.53,1.1119e6,0.0,0.0
"""max""",743.0,"""TRANSFER""",9.2446e7,"""C999999784""",5.9585e7,4.9585e7,"""M999999784""",3.5602e8,3.5618e8,1.0,1.0


### **Insights**


* **Flawless Data Quality (No Nulls):** The null_count row shows 0.0 for absolutely all columns. There will be no need to apply missing data imputation techniques to this original dataset.

* **Extreme Target Variable Imbalance:** The mean of the isFraud column is only 0.001291. This indicates that **only ~0.13% of the transactions** in the dataset represent confirmed frauds. Future Machine Learning models will require balancing techniques.

* **Inefficiency of the Current Business Rule (isFlaggedFraud):** The current system's flagging mean (0.000003) is significantly lower than the actual frauds, indicating that the fixed rule (transfers > 200,000) fails to capture malicious behavior. This provides strong technical and commercial justification for building a robust predictive model.

* **Asymmetry and Outliers in Transaction Values (amount):** The transaction amount column is highly right-skewed. While the 3rd quartile (75%) concentrates transactions up to ~208k, the maximum value reaches over **92 million**. The mean (~179k) is considerably pulled up by these extreme values compared to the median (~74k).


In [5]:
df.head()

step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
i64,str,f64,str,f64,f64,str,f64,f64,i64,i64
1,"""PAYMENT""",9839.64,"""C1231006815""",170136.0,160296.36,"""M1979787155""",0.0,0.0,0,0
1,"""PAYMENT""",1864.28,"""C1666544295""",21249.0,19384.72,"""M2044282225""",0.0,0.0,0,0
1,"""TRANSFER""",181.0,"""C1305486145""",181.0,0.0,"""C553264065""",0.0,0.0,1,0
1,"""CASH_OUT""",181.0,"""C840083671""",181.0,0.0,"""C38997010""",21182.0,0.0,1,0
1,"""PAYMENT""",11668.14,"""C2048537720""",41554.0,29885.86,"""M1230701703""",0.0,0.0,0,0


In [6]:
numeric_columns = [
    'step',
    'amount',
    'oldbalanceOrg',
    'newbalanceOrig',
    'oldbalanceDest',
    'newbalanceDest'
]

categorical_columns = [
    'type'
]

boolean_columns = [
    'isFraud',
    'isFlaggedFraud'
]

id_columns = [
    'nameOrig', 
    'nameDest'
]


In [9]:
vis.grafico_frequencia_percentual_alvo(df.to_pandas(), 'type')

In [12]:
vis.grafico_frequencia_percentual_alvo(df.to_pandas(), 'type', hue='isFraud', mapa_cores={0:'green', 1:'red'})